# 00 — Complete ETL Pipeline: Bronze → Silver → Gold

**Purpose**: Execute the full data engineering pipeline to materialize Bronze, Silver, and Gold layer datasets from raw sources.

**Prerequisites**:
- AWS credentials configured (`aws configure --profile mba-thesis`)
- API keys in `.env` (for Bronze ingestion from APIs)
- Python dependencies installed (`pip install -r requirements.txt`)

**Outputs**:
- Bronze layer: Raw API responses and CSV extracts
- Silver layer: Normalized, typed, deduplicated datasets
- Gold layer: Analysis-ready aggregations for downstream notebooks (01-06)

---

## Architecture Overview

This notebook implements the **Medallion Architecture** (Databricks, 2023):

```
┌─────────────────────────────────────────────────────────────────┐
│  BRONZE LAYER (Raw)                                             │
│  • IBGE API responses (Censo 2010, 2022)                        │
│  • Portal da Transparência (transfers, sanctions)               │
│  • IPCA/BCB deflator series                                     │
├─────────────────────────────────────────────────────────────────┤
│  SILVER LAYER (Normalized)                                      │
│  • Star schema with dimension + fact tables                     │
│  • Type enforcement, deduplication, deflation                   │
│  • 7-digit IBGE municipal code standard                         │
├─────────────────────────────────────────────────────────────────┤
│  GOLD LAYER (Analysis-Ready)                                    │
│  • Consolidated municipal socioeconomic profiles                │
│  • State-level aggregations                                     │
│  • Clustering features (normalized)                             │
│  • Analysis datasets for ML and statistics                      │
└─────────────────────────────────────────────────────────────────┘
```

**Downstream Notebooks**: After running this ETL, proceed to:
- `01_exploratory_data_analysis.ipynb` — EDA on Gold datasets
- `02_statistical_analysis.ipynb` — OLS regression
- `03_machine_learning.ipynb` — Predictive modeling
- `04_clustering_analysis.ipynb` — K-means segmentation
- `05_corruption_hdi_clusters.ipynb` — Corruption vs HDI analysis
- `06_complete_thesis_pipeline.ipynb` — Master thesis notebook (reruns ETL + all analyses)


In [10]:
# --- AUTO-GENERATED DEPENDENCY INSTALL ---
# Installs all project dependencies on first run (Colab, fresh environments, etc).
# Idempotent: pip skips anything already installed.
# To regenerate this cell, run: python scripts/inject_pip_install.py

import subprocess
import sys
from pathlib import Path

_req = Path.cwd().parent / "requirements.txt"
if not _req.exists():
    _req = Path.cwd() / "requirements.txt"

if _req.exists():
    print(f"Installing dependencies from {_req.name if _req.exists() else "requirements.txt"} ...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", str(_req)])
    print("Dependencies ready.")
else:
    print("requirements.txt not found. Install manually: pip install -r requirements.txt")


Installing dependencies from requirements.txt ...
Dependencies ready.


---

## 1. Environment Setup

### 1.1 Install Dependencies (if needed)


In [11]:
# Auto-install dependencies on first run
import subprocess
import sys
from pathlib import Path

_req = Path.cwd().parent / "requirements.txt"
if not _req.exists():
    _req = Path.cwd() / "requirements.txt"

if _req.exists():
    print(f"Installing dependencies from {_req.name if _req.exists() else "requirements.txt"} ...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", str(_req)])
    print("Dependencies ready.")
else:
    print("requirements.txt not found. Install manually: pip install -r requirements.txt")


Installing dependencies from requirements.txt ...
Dependencies ready.


### 1.2 Imports and Configuration


In [12]:
import os
import sys
import json
import warnings
from pathlib import Path
from datetime import datetime

import pandas as pd
import numpy as np

# Add project root to path
sys.path.insert(0, str(Path.cwd().parent))

# Project modules
from src.ingestion.ibge_client import IBGEIngestor
from src.ingestion.transparency_client import TransparencyIngestor
from src.processing.ibge_transformer import IBGETransformer
from src.processing.transparency_transformer import TransparencyTransformer
from src.processing.gold_transformer import GoldTransformer
from src.config.runtime_config import load_runtime_config

print(f"Python path: {(sys.path[0] if sys.path[0] == "." else "<project-root>")}")
print(f"Working directory: {Path.cwd().name if Path.cwd().name else "<root>"}")
print(f"Timestamp: {datetime.now().isoformat()}")


Python path: <project-root>
Working directory: notebooks
Timestamp: 2026-04-19T17:58:19.769600


### 1.3 Reproducibility Settings


In [13]:
# Fixed seed for reproducibility
SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)

import random
random.seed(SEED)
np.random.seed(SEED)

print(f"Reproducibility seed fixed at {SEED}")
print(f"NumPy version: {np.__version__}")
print(f"Pandas version: {pd.__version__}")


Reproducibility seed fixed at 42
NumPy version: 2.4.4
Pandas version: 3.0.2


---

## 2. Bronze Layer — Raw Data Ingestion

### 2.1 Methodology

The Bronze layer preserves raw data exactly as received from sources (Armbrust et al., 2020). This ensures:
- **Auditability**: Original data can be re-examined
- **Reproducibility**: Pipeline can be replayed from raw state
- **Legal compliance**: LAI (2011) and LGPD (2018) requirements for data lineage

**Data Sources**:
- IBGE SIDRA API: Municipal demographics (Censo 2010, 2022)
- Portal da Transparência: Federal transfers and sanctions
- BCB SGS: IPCA deflator series

### 2.2 Load Runtime Configuration


In [14]:
# Load environment variables from .env first (highest priority)
from dotenv import load_dotenv
env_path = Path("../.env")
if env_path.exists():
    load_dotenv(env_path)
    print(f"Loaded environment from {env_path}")
else:
    print("WARNING: .env file not found")

# Load runtime configuration (lower priority than .env)
config_path = Path("../config/runtime_config.json")
if config_path.exists():
    with open(config_path) as f:
        runtime_config = json.load(f)
else:
    runtime_config = {}

# Extract settings - .env takes precedence
aws_profile = os.environ.get("AWS_PROFILE") or runtime_config.get("aws", {}).get("profile", "mba-thesis")
s3_bucket = os.environ.get("S3_BUCKET_NAME") or runtime_config.get("aws", {}).get("s3_bucket_name", "enok-mba-thesis-datalake")
use_local_cache = runtime_config.get("execution", {}).get("use_local_cache", True)

# Set AWS_PROFILE so boto3 uses it
os.environ["AWS_PROFILE"] = aws_profile

print(f"AWS Profile: {aws_profile}")
print(f"S3 Bucket: {s3_bucket}")
print(f"Use Local Cache: {use_local_cache}")


Loaded environment from ..\.env
AWS Profile: mba-thesis
S3 Bucket: enok-mba-thesis-datalake
Use Local Cache: True


### 2.3 Execute Bronze Ingestion

**Note**: Bronze ingestion from APIs requires:
- IBGE: No API key needed (public data)
- Portal da Transparência: `TRANSPARENCY_API_KEY` in `.env`

To skip API calls and use existing local data, set:
```python
SKIP_BRONZE_INGESTION = True
```


In [15]:
# ===== Bronze Layer: Sync from S3 to local =====
# Uses src.ingestion.s3_local_sync to mirror s3://$S3_BUCKET/bronze/ into data/bronze/.
# Idempotent: only downloads files that are missing locally or have different sizes.
from src.ingestion.s3_local_sync import sync_s3_prefix_to_local

local_data_dir = Path("../data")
bronze_dir = local_data_dir / "bronze"

s3_bucket = os.getenv("S3_BUCKET_NAME", "enok-mba-thesis-datalake")
print(f"Syncing s3://{s3_bucket}/bronze/ -> {bronze_dir}")

result = sync_s3_prefix_to_local(
    bucket=s3_bucket,
    prefix="bronze/",
    local_dir=bronze_dir,
)
print(f"[SYNC] downloaded={result['downloaded']} skipped={result['skipped']} errors={result['errors']}")

bronze_files = list(bronze_dir.rglob("*.json")) + list(bronze_dir.rglob("*.parquet"))
by_source = {}
for f in bronze_files:
    by_source.setdefault(f.parent.name, []).append(f)
print(f"\n[OK] Bronze layer: {len(bronze_files)} files total")
for source, files in sorted(by_source.items()):
    print(f"  - {source}: {len(files)} file(s)")


2026-04-19 17:58:19,891 - INFO - Found credentials in shared credentials file: ~/.aws/credentials


Syncing s3://enok-mba-thesis-datalake/bronze/ -> ..\data\bronze


2026-04-19 17:58:26,323 - INFO - [sync] s3://enok-mba-thesis-datalake/bronze/ -> ..\data\bronze : already up to date (282 files)


[SYNC] downloaded=0 skipped=282 errors=0

[OK] Bronze layer: 282 files total
  - .metadata: 159 file(s)
  - economic: 1 file(s)
  - ibge: 10 file(s)
  - transparency: 112 file(s)


---

## 3. Silver Layer — Normalization and Transformation

### 3.1 Methodology

Silver layer applies:
- **Schema enforcement**: Type validation per `config/silver_schemas.json`
- **Deflation**: Nominal → Real BRL using IPCA (base 2022)
- **Deduplication**: Remove duplicates from incremental loads
- **Star schema**: Dimension tables + fact tables (Kimball & Ross, 2013)

**Transformation Pipeline**:
1. Load Bronze raw data
2. Apply type mappings and validations
3. Deflate monetary values (IPCA)
4. Standardize municipal codes (7-digit IBGE)
5. Write Silver Parquet files


### 3.2 Execute Silver Transformations


In [16]:
# ===== Silver Layer: Sync from S3 to local =====
# If Silver is missing on S3, run scripts/02_silver_transformation.sh first.
silver_dir = local_data_dir / "silver"
print(f"Syncing s3://{s3_bucket}/silver/ -> {silver_dir}")

result = sync_s3_prefix_to_local(
    bucket=s3_bucket,
    prefix="silver/",
    local_dir=silver_dir,
)
print(f"[SYNC] downloaded={result['downloaded']} skipped={result['skipped']} errors={result['errors']}")

silver_files = list(silver_dir.rglob("*.parquet"))
if not silver_files:
    print("\n[WARN] Silver layer is empty on S3 too. Run: bash scripts/02_silver_transformation.sh")
else:
    datasets = {}
    for f in silver_files:
        datasets.setdefault(f.parent.name, []).append(f)
    print(f"\n[OK] Silver layer: {len(silver_files)} Parquet file(s)")
    for dataset, files in sorted(datasets.items()):
        print(f"  - {dataset}: {len(files)} file(s)")


2026-04-19 17:58:26,380 - INFO - Found credentials in shared credentials file: ~/.aws/credentials


Syncing s3://enok-mba-thesis-datalake/silver/ -> ..\data\silver


2026-04-19 17:58:29,250 - INFO - [sync] s3://enok-mba-thesis-datalake/silver/ -> ..\data\silver : already up to date (27 files)


[SYNC] downloaded=0 skipped=27 errors=0

[OK] Silver layer: 9 Parquet file(s)
  - dim_inflation_index: 1 file(s)
  - dim_municipalities: 1 file(s)
  - dim_municipality_lookup: 1 file(s)
  - fact_federal_transfers: 1 file(s)
  - fact_income: 1 file(s)
  - fact_literacy: 1 file(s)
  - fact_population: 1 file(s)
  - fact_sanctions: 1 file(s)
  - fact_sanitation: 1 file(s)


---

## 4. Gold Layer — Analysis-Ready Aggregations

### 4.1 Methodology

Gold layer creates denormalized, analysis-optimized datasets (Kleppmann, 2017):
- **Municipal profiles**: Socioeconomic indicators + deltas (2010→2022)
- **State summaries**: Aggregated by UF (27 states)
- **Clustering features**: Normalized vectors for K-means
- **ML datasets**: Labelled datasets for supervised learning

**Key Datasets Produced**:
| Dataset | Records | Purpose |
|---------|---------|---------|
| `agg_municipality_socioeconomic` | ~5,570 | Municipal feature vectors |
| `agg_state_summary` | 27 | State-level aggregations |
| `analysis_compliance` | 27 | ML-ready state dataset |
| `analysis_compliance_municipality` | ~5,570 | ML-ready municipal dataset |
| `consolidated_clustering` | ~5,565 | Normalized clustering features |


### 4.2 Execute Gold Transformations


In [17]:
# ===== Gold Layer: Sync from S3 to local =====
# If Gold is missing on S3, run scripts/03_gold_transformation.sh first.
gold_dir = local_data_dir / "gold"
print(f"Syncing s3://{s3_bucket}/gold/ -> {gold_dir}")

result = sync_s3_prefix_to_local(
    bucket=s3_bucket,
    prefix="gold/",
    local_dir=gold_dir,
)
print(f"[SYNC] downloaded={result['downloaded']} skipped={result['skipped']} errors={result['errors']}")

gold_files = list(gold_dir.rglob("*.parquet"))
if not gold_files:
    print("\n[WARN] Gold layer is empty on S3 too. Run: bash scripts/03_gold_transformation.sh")
else:
    datasets = {}
    for f in gold_files:
        datasets.setdefault(f.parent.name, []).append(f)
    print(f"\n[OK] Gold layer: {len(gold_files)} Parquet file(s)")
    for dataset, files in sorted(datasets.items()):
        print(f"  - {dataset}: {len(files)} file(s)")
print("\n[OK] Gold data available - ready for analysis notebooks (01-06)")


2026-04-19 17:58:29,285 - INFO - Found credentials in shared credentials file: ~/.aws/credentials


Syncing s3://enok-mba-thesis-datalake/gold/ -> ..\data\gold


2026-04-19 17:58:30,458 - INFO - [sync] s3://enok-mba-thesis-datalake/gold/ -> ..\data\gold : already up to date (22 files)


[SYNC] downloaded=0 skipped=22 errors=0

[OK] Gold layer: 6 Parquet file(s)
  - agg_municipality_socioeconomic: 1 file(s)
  - agg_sanctions_summary: 1 file(s)
  - agg_state_summary: 1 file(s)
  - analysis_compliance: 1 file(s)
  - analysis_compliance_municipality: 1 file(s)
  - consolidated_clustering: 1 file(s)

[OK] Gold data available - ready for analysis notebooks (01-06)


---

## 5. ETL Validation & Quality Checks

### 5.1 Row Counts Across Layers


In [18]:
def count_layer_rows(layer_path: Path) -> dict:
    # Count rows in all Parquet files in a layer
    counts = {}
    if not layer_path.exists():
        return counts
    
    for parquet_file in layer_path.rglob("*.parquet"):
        try:
            df = pd.read_parquet(parquet_file)
            dataset = parquet_file.parent.name
            counts[dataset] = len(df)
        except Exception:
            pass
    return counts

bronze_counts = count_layer_rows(Path("../data/bronze"))
silver_counts = count_layer_rows(Path("../data/silver"))
gold_counts = count_layer_rows(Path("../data/gold"))

print("=" * 60)
print("ETL LAYER SUMMARY")
print("=" * 60)
print(f"\nBRONZE: {len(bronze_counts)} datasets, {sum(bronze_counts.values()):,} total rows")
print(f"SILVER: {len(silver_counts)} datasets, {sum(silver_counts.values()):,} total rows")
print(f"GOLD:   {len(gold_counts)} datasets, {sum(gold_counts.values()):,} total rows")

print("\n" + "=" * 60)
print("GOLD DATASETS (Downstream Analysis Ready)")
print("=" * 60)
for dataset, count in sorted(gold_counts.items()):
    print(f"  • {dataset}: {count:,} rows")


ETL LAYER SUMMARY

BRONZE: 0 datasets, 0 total rows
SILVER: 9 datasets, 85,541 total rows
GOLD:   6 datasets, 16,762 total rows

GOLD DATASETS (Downstream Analysis Ready)
  • agg_municipality_socioeconomic: 5,570 rows
  • agg_sanctions_summary: 3 rows
  • agg_state_summary: 27 rows
  • analysis_compliance: 27 rows
  • analysis_compliance_municipality: 5,570 rows
  • consolidated_clustering: 5,565 rows


---

## 6. Next Steps

The ETL pipeline is now complete. Gold layer datasets are ready for analysis.

### Recommended Notebook Sequence:

1. **`00_etl_pipeline.ipynb`** ← You are here (ETL complete)
2. **`01_exploratory_data_analysis.ipynb`** — EDA, distributions, quality checks
3. **`02_statistical_analysis.ipynb`** — OLS regression, correlations
4. **`03_machine_learning.ipynb`** — Predictive models (ElasticNet, Random Forest)
5. **`04_clustering_analysis.ipynb`** — K-means segmentation with PCA
6. **`05_corruption_hdi_clusters.ipynb`** — Corruption vs HDI cluster-stratified analysis
7. **`06_complete_thesis_pipeline.ipynb`** — Master notebook (reruns ETL + all analyses)

### Or run the shell scripts directly:

```bash
# Complete pipeline from scratch (requires API keys)
./scripts/01_bronze_ingestion.sh
./scripts/02_silver_transformation.sh
./scripts/03_gold_transformation.sh

# Or all at once
./scripts/run_pipeline.sh
```

### Storage Locations:
- Local: `data/bronze/`, `data/silver/`, `data/gold/`
- S3: `s3://{s3_bucket}/bronze/`, `/silver/`, `/gold/`


---

## References

- **Armbrust, M. et al. (2020).** Lakehouse: A New Generation of Open Platforms. Databricks.
- **Databricks (2023).** Medallion Architecture: Best Practices.
- **Kimball, R.; Ross, M. (2013).** The Data Warehouse Toolkit. 3rd ed. Wiley.
- **Kleppmann, M. (2017).** Designing Data-Intensive Applications. O'Reilly.
- **Brasil (2011).** Lei nº 12.527/2011 — Lei de Acesso à Informação.
- **Brasil (2018).** Lei nº 13.709/2018 — LGPD.
